In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA




daily = pd.read_parquet("data/1_prepared_views/prepared_daily_view.parquet")
print(daily.shape)   # (56110, 44): one row per machine per day
print(daily.head())
daily.columns 

(56110, 44)
  machine_id gaming_date    month local_weekday  is_weekend federal_holiday  \
0     M-1001  2025-01-01  2025-01           Wed       False  New Year's Day   
1     M-1001  2025-01-02  2025-01           Thu       False            None   
2     M-1001  2025-01-03  2025-01           Fri       False            None   
3     M-1001  2025-01-04  2025-01           Sat        True            None   
4     M-1001  2025-01-05  2025-01           Sun        True            None   

  events_active zone_id      zone_name bank_id  ... zone_entries_observed  \
0          None   Z-ENT  Main Entrance    B-01  ...                1999.0   
1          None   Z-ENT  Main Entrance    B-01  ...                2530.0   
2          None   Z-ENT  Main Entrance    B-01  ...                3145.0   
3          None   Z-ENT  Main Entrance    B-01  ...                3560.0   
4          None   Z-ENT  Main Entrance    B-01  ...                2865.0   

  zone_counter_coverage_hours lease_cost_usd servi

Index(['machine_id', 'gaming_date', 'month', 'local_weekday', 'is_weekend',
       'federal_holiday', 'events_active', 'zone_id', 'zone_name', 'bank_id',
       'location_id', 'config_id', 'theme', 'configured_rtp',
       'volatility_class', 'manufacturer', 'lease_class', 'active',
       'meter_status', 'available_hours', 'occupied_hours',
       'availability_ratio', 'occupancy_ratio', 'paid_spins', 'coin_in_usd',
       'payout_usd', 'house_win_usd', 'theo_usd', 'observed_hold', 'theo_hold',
       'luck_residual_usd', 'carded_coin_in_usd', 'carded_share',
       'faults_opened', 'zone_entries_observed', 'zone_counter_coverage_hours',
       'lease_cost_usd', 'service_cost_usd', 'power_cost_usd',
       'repair_cost_usd', 'contribution_after_supplied_costs_usd',
       'lag_coin_in_7d_mean_usd', 'lag_downtime_7d_hours',
       'lag_house_win_28d_usd'],
      dtype='object')

In [32]:
len(pd.unique(daily['machine_id']))

310

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# =====================================================================
# 1. FEATURE SELECTION: Isolate Exposure, Availability & Configuration
# =====================================================================
operational_features = [
    'zone_id', 'zone_name', 'bank_id', 'config_id', 'theme', 
    'configured_rtp', 'volatility_class', 'manufacturer', 'lease_class',
    'available_hours', 'availability_ratio', 'zone_entries_observed', 
    'local_weekday', 'is_weekend', 'federal_holiday'
]

meta_columns = ['machine_id', 'contribution_after_supplied_costs_usd', 'luck_residual_usd']

df_clean = daily[operational_features + meta_columns].copy()

# Fix the data types immediately by casting the targets to standard float
df_clean['contribution_after_supplied_costs_usd'] = df_clean['contribution_after_supplied_costs_usd'].astype(float)
df_clean['luck_residual_usd'] = df_clean['luck_residual_usd'].astype(float)

X = df_clean[operational_features]
y = df_clean['contribution_after_supplied_costs_usd']

# =====================================================================
# 2. ENCODING & IMPUTATION (Memory Efficient)
# =====================================================================
object_cols = X.select_dtypes(include=['object', 'category']).columns
X_encoded = X.copy()
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_encoded[object_cols] = encoder.fit_transform(X[object_cols].astype(str))

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X_encoded), columns=X_encoded.columns)

# =====================================================================
# 3. TRAIN THE EXPECTATION MODEL
# =====================================================================
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_imputed, y)

# =====================================================================
# 4. RESIDUAL BREAKDOWN: Expected vs. Actual Performance
# =====================================================================
# Predict what the machine "should" have made (this returns a float array)
df_clean['Expected_Contribution'] = rf_model.predict(X_imputed)

# Math works perfectly now because both series are float64 types
df_clean['Model_Residual'] = df_clean['contribution_after_supplied_costs_usd'] - df_clean['Expected_Contribution']

# =====================================================================
# 5. DIAGNOSTIC AGGREGATION BY MACHINE ID
# =====================================================================
machine_analysis = df_clean.groupby('machine_id').agg({
    'contribution_after_supplied_costs_usd': 'sum',
    'Expected_Contribution': 'sum',
    'Model_Residual': 'sum',      
    'luck_residual_usd': 'sum',    
    'availability_ratio': 'mean',  
    'zone_entries_observed': 'sum' 
}).reset_index()

# Math works perfectly here too
machine_analysis['Structural_Leak_Score'] = machine_analysis['Model_Residual'] - machine_analysis['luck_residual_usd']

# =====================================================================
# 6. IDENTIFY THE TRUE "STRUCTURAL LEAKS"
# =====================================================================
true_leaks = machine_analysis.sort_values(by='Structural_Leak_Score', ascending=True)

print("Top 10 True Structural Leaks (Underperforming despite Traffic, Uptime, and Luck):")
print(true_leaks[['machine_id', 'Model_Residual', 'luck_residual_usd', 'Structural_Leak_Score']].head(10))

print("\nTop 10 Unlucky Machines (Low performance completely explained by bad luck):")
unlucky_machines = machine_analysis.sort_values(by='luck_residual_usd', ascending=True)
print(unlucky_machines[['machine_id', 'Model_Residual', 'luck_residual_usd', 'Structural_Leak_Score']].head(10))


Top 10 True Structural Leaks (Underperforming despite Traffic, Uptime, and Luck):
    machine_id  Model_Residual  luck_residual_usd  Structural_Leak_Score
266     M-1267     7792.827633          60587.480          -52794.652367
213     M-1214     5501.063563          36494.000          -30992.936437
211     M-1212    -2894.184800          24061.920          -26956.104800
86      M-1087     3741.857833          28353.600          -24611.742167
168     M-1169     4856.516800          28482.248          -23625.731200
169     M-1170     1350.663895          23027.232          -21676.568105
124     M-1125     5442.660024          26640.500          -21197.839976
26      M-1027     6139.889176          25853.120          -19713.230824
197     M-1198    11228.794933          30318.520          -19089.725067
171     M-1172     1576.082663          19899.900          -18323.817337

Top 10 Unlucky Machines (Low performance completely explained by bad luck):
    machine_id  Model_Residual  luck_r

In [47]:
# 1. Isolate the expanded list of 10 true leak IDs
leak_ids = [
    'M-1267', 'M-1214', 'M-1212', 'M-1087', 'M-1169', 
    'M-1170', 'M-1125', 'M-1027', 'M-1198', 'M-1172'
]

# 2. Filter daily dataset down to these specific profiles
leak_profiles = daily[daily['machine_id'].isin(leak_ids)].copy()

# 3. Pull primary configurations to audit them (taking first static setup row)
profile_summary = leak_profiles.groupby('machine_id').agg({
    'theme': 'first',
    'zone_name': 'first',
    'configured_rtp': 'first',
    'volatility_class': 'first',
    'manufacturer': 'first',
    'config_id': 'first',
    'bank_id': 'first'
}).reset_index()

# 4. Sort the summary to match the sequence of your leak severity list
profile_summary['machine_id'] = pd.Categorical(profile_summary['machine_id'], categories=leak_ids, ordered=True)
profile_summary = profile_summary.sort_values('machine_id')

print("Structural Auditing Profile for Top 10 Leaking Machines:")
print(profile_summary.to_string(index=False))


Structural Auditing Profile for Top 10 Leaking Machines:
machine_id          theme       zone_name configured_rtp volatility_class manufacturer config_id bank_id
    M-1267 Jackpot Summit   Main Entrance     0.88000000        very_high    Ridgeline   G-JKP88    B-06
    M-1214 Platinum Reels High-Limit Room     0.90000000             high    Ridgeline    G-HL90    B-27
    M-1212 Jackpot Summit      Center Pit     0.88000000        very_high    Northwind   G-JKP88    B-16
    M-1087 Jackpot Summit      Center Pit     0.88000000        very_high    Northwind   G-JKP88    B-15
    M-1169    Neon Nights  Hotel Corridor     0.88000000             high    Northwind   G-NEO88    B-39
    M-1170    Neon Nights   Main Entrance     0.88000000             high    Northwind   G-NEO88    B-03
    M-1125 Platinum Reels High-Limit Room     0.90000000             high    Ridgeline    G-HL90    B-29
    M-1027 Jackpot Summit         Bar Row     0.88000000        very_high    Ridgeline   G-JKP88    B-2